# Heteroscedasticity: lab notebook

**Statistical Modelling and Forecasting, Foundations of time series**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
from statsmodels.stats.diagnostic import het_arch
from scipy import stats

### 1. Simulate an ARCH(1) series: sigma_t^2 = a0 + a1 * e_{t-1}^2

In [ ]:
# 1. Simulate an ARCH(1) series: sigma_t^2 = a0 + a1 * e_{t-1}^2
rng = np.random.default_rng(11)
n, a0, a1 = 2000, 0.2, 0.6
y = np.zeros(n)
for t in range(1, n):
    sig2 = a0 + a1 * y[t - 1] ** 2
    y[t] = np.sqrt(sig2) * rng.normal()
print("sample variance:", round(y.var(), 3), "| theory a0/(1-a1):", round(a0 / (1 - a1), 3))
print("kurtosis (normal = 3):", round(stats.kurtosis(y, fisher=False), 2))

### 2. ARCH-LM test = n * R^2 of squared values on their lags

In [ ]:
# 2. ARCH-LM test = n * R^2 of squared values on their lags
lm, p, f, pf = het_arch(y, nlags=4)
print("ARCH-LM (4 lags):", round(lm, 1), " p =", round(p, 6))
white = rng.normal(size=n)
lm0, p0, _, _ = het_arch(white, nlags=4)
print("white noise ARCH-LM:", round(lm0, 2), " p =", round(p0, 3))

### 3. The numerical problem: forecasts of the conditional variance

In [ ]:
# 3. The numerical problem: forecasts of the conditional variance
a0, a1, eps = 0.4, 0.5, -2.0
h1 = a0 + a1 * eps ** 2
h2 = a0 + a1 * h1
print("sigma^2(T+1) =", h1, "| sigma^2(T+2) =", h2, "| unconditional =", a0 / (1 - a1))
chi2_crit = stats.chi2.ppf(0.95, 4)
print("chi2(4) 5% critical value:", round(chi2_crit, 2), "| LM for n=200, R2=0.05 is", 200 * 0.05)

### 4. A growing-level series: the log makes the spread constant

In [ ]:
# 4. A growing-level series: the log makes the spread constant
t = np.arange(400)
level = 100 * np.exp(0.006 * t)
x = level * (1 + 0.08 * rng.normal(size=400))
for name, s in [("raw", x), ("log", np.log(x))]:
    res = s - np.polyval(np.polyfit(t, s, 1), t)
    print(f"{name}: residual sd first third {res[:133].std():.3f}, last third {res[-133:].std():.3f}")

## Expected output

Running all cells above should print:

```
sample variance: 0.487 | theory a0/(1-a1): 0.5
kurtosis (normal = 3): 6.58
ARCH-LM (4 lags): 412.5  p = 0.0
white noise ARCH-LM: 2.49  p = 0.647
sigma^2(T+1) = 2.4 | sigma^2(T+2) = 1.6 | unconditional = 0.8
chi2(4) 5% critical value: 9.49 | LM for n=200, R2=0.05 is 10.0
raw: residual sd first third 53.618, last third 98.669
log: residual sd first third 0.079, last third 0.079
```

## Practice exercises

Try these in the cells below.

1. Raise α₁ in the ARCH simulator. What happens to the clustering and to the kurtosis?
2. Choose "Growing level (log fixes it)", tick the log transform, and compare the spread before and after.
3. For α₀ = 0.2 and α₁ = 0.5 find the unconditional variance, then check it against the simulated variance.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
